# 🔮 FORESIGHT — 03: Leakage-Safe Feature Engineering Pipeline

This notebook demonstrates the mathematical feature engineering pipeline: autoregressive lags, causal rolling statistics, calendar seasonality encodings, velocity momentum ratios, and commercial pricing signals.

In [ ]:
import pandas as pd
import numpy as np
from foresight.data.loader import load_processed_sales
from foresight.features.pipeline import FeatureEngineeringPipeline

# 1. Load Clean Processed Dataset
df = load_processed_sales()
print(f"Loaded processed sales: {len(df):,} observations")

## 2. Execute Feature Engineering Pipeline
Apply strictly causal transformations with warmup horizon truncation.

In [ ]:
pipeline = FeatureEngineeringPipeline(dropna_warmup=True)
features_df = pipeline.fit_transform(df)

meta = pipeline.get_feature_metadata(features_df)
print(f"Generated {meta.total_features} predictive features across {len(features_df):,} rows.")
print("Predictive Features:", meta.feature_names)

## 3. Autoregressive Lag Correlation Analysis
Inspect autocorrelation between demand lags (1d, 7d, 14d, 28d, 56d) and contemporaneous demand.

In [ ]:
lag_cols = [c for c in features_df.columns if c.startswith("lag_")]
corr = features_df[["quantity"] + lag_cols].corr()["quantity"].sort_values(ascending=False)
print("Correlation with Quantity:")
print(corr)

## 4. Verification of Zero Target Leakage
Ensure future sales are never reflected in historical features.

In [ ]:
# Mathematical verification that rolling_mean_7 at time t matches manual average over [t-7, t-1]
sample_series = features_df[(features_df["sku_id"] == "SKU-1001") & (features_df["store_id"] == "STORE-001")].copy().reset_index(drop=True)

for idx in range(10, 15):
    pipeline_rolling_mean = sample_series.loc[idx, "rolling_mean_7"]
    manual_mean = sample_series.loc[idx-7:idx-1, "quantity"].mean()
    assert np.isclose(pipeline_rolling_mean, manual_mean, atol=1e-3)
    print(f"Row {idx} (Date {sample_series.loc[idx, 'date']}): Pipeline={pipeline_rolling_mean:.2f}, Manual={manual_mean:.2f} [MATCH]")

print("\nZero Target Leakage Mathematically Verified! ✅")